# Fine-tune T5-base on ParaDetox (SFT Stage)

Pipeline:
1. Install dependencies + mount Google Drive
2. Load `s-nlp/paradetox` dataset from HuggingFace, preview sample rows
3. Use all 19,744 samples, split train/validation (90%/10%)
4. Tokenize inputs with prefix `"detoxify: "` (max_source_length = max_target_length = 256)
5. Train T5-base with early stopping (patience=3, val loss)
6. Save best full model to Google Drive


## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece evaluate


## 2. Environment & Output Directory Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Output directory will be created in MyDrive
OUTPUT_DIR = "/content/drive/MyDrive/Data_Khoaluan/T5_base"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Model will be saved at: {OUTPUT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model will be saved at: /content/drive/MyDrive/Data_Khoaluan/T5


## 3. Load ParaDetox Dataset from HuggingFace

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("s-nlp/paradetox")
print(raw_dataset)
print()
print("First few rows of training set:")
for i in range(5):
    example = raw_dataset["train"][i]
    print(f"[{i}] toxic   : {example['en_toxic_comment']}")
    print(f"    neutral : {example['en_neutral_comment']}")
    print()


DatasetDict({
    train: Dataset({
        features: ['en_toxic_comment', 'en_neutral_comment'],
        num_rows: 19744
    })
})

First few rows of training set:
[0] toxic   : he had steel balls too !
    neutral : he was brave too!

[1] toxic   : dude should have been taken to api , he would be right at home with all the other knuckleheads there
    neutral : It would have been good if he went to api. He would fit in.

[2] toxic   : im not gonna sell the fucking picture , i just want to contribute to the fucking article .
    neutral : I'm not gonna sell the picture, i just want to contribute to the article.

[3] toxic   : the garbage that is being created by cnn and other news agencies is outrageous .
    neutral : the news that is being created by cnn and other news agencies is outrageous.

[4] toxic   : the reason they dont exist is because neither is a pathological liar like trump .
    neutral : The reason they don't exist is because neither lies like trump



## 4. Dataset Partition: Train / Validation

- Full 19,744 samples — 90% train / 10% validation
- Random shuffle with fixed seed for exact reproducibility


In [ ]:
SEED = 42
VAL_RATIO = 0.1  # ~1,974 val samples, ~17,770 train samples

shuffled = raw_dataset["train"].shuffle(seed=SEED)

split_dataset = shuffled.train_test_split(test_size=VAL_RATIO, seed=SEED)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"Train: {len(train_dataset)} dong")
print(f"Validation: {len(val_dataset)} dong")
print()
print("Vi du 1 dong trong tap train:")
print(train_dataset[0])


Train: 4500 dong
Validation: 500 dong

Vi du 1 dong trong tap train:
{'en_toxic_comment': 'how fucking bizarre was that ?', 'en_neutral_comment': 'How strange was that?'}


## 5. Tokenization & Preprocessing

- Model: `t5-base`
- Task prefix: `"detoxify: "`
- max_source_length = max_target_length = 256
- Label padding token replaced by -100 to ignore padding in loss computation


In [ ]:
from transformers import T5TokenizerFast

MODEL_NAME = "t5-base"
TASK_PREFIX = "detoxify: "
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 256

tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    inputs = [TASK_PREFIX + text for text in examples["en_toxic_comment"]]
    targets = examples["en_neutral_comment"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LEN,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train set",
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation set",
)

print(tokenized_train)
print(tokenized_val)


Tokenizing validation set:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4500
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 500
})


## 6. Initialize T5-base Model

In [ ]:
from transformers import T5ForConditionalGeneration
import torch

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


[transformers] loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--t5-base/snapshots/a9723ea7f1b39c1eae772870f3b547bf6ef7e6c1/config.json
[transformers] Model config T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 3072,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "relu",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "relu",
  "initializer_factor": 1.0,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "is_gated_act": false,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "scale_decoder_outputs": true,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

[transformers] loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--t5-base/snapshots/a9723ea7f1b39c1eae772870f3b547bf6ef7e6c1/generation_config.json
[transformers] Generate config GenerationConfig {
  "decoder_start_token_id": 0,
  "eos_token_id": 1,
  "pad_token_id": 0
}



Device: cuda
GPU: Tesla T4


## 7. Data Collator (Dynamic Batch Padding)

Dynamic batch padding instead of fixed 256 padding to optimize compute efficiency.


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
)


## 8. Training Arguments & Trainer Initialization

| Parameter | Value |
|---|---|
| batch_size | 4 |
| gradient_accumulation_steps | 8 (effective batch = 32) |
| learning_rate | 2e-4 |
| Max Epochs | 15 |
| fp16 | True |
| eval + save | per epoch |
| Early Stopping | Patience = 3 (eval_loss) |
| Load Best Model at End | True |


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback
import math

CHECKPOINT_DIR = "/content/t5_paradetox_checkpoints"  # Local checkpoint scratch directory

# The Trainer will calculate warmup steps based on warmup_ratio
# len(train_dataset) is 4500
# per_device_train_batch_size is 4
# gradient_accumulation_steps is 8
# num_train_epochs is 15

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    num_train_epochs=15,
    # Removed warmup_ratio due to TypeError
    lr_scheduler_type="linear",
    fp16=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=3,
    report_to="none",
    disable_tqdm=False,   # dam bao progress bar hien thi
    log_level="info",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    # processing_class=tokenizer, # This argument is not valid for Seq2SeqTrainer
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)


[transformers] PyTorch: setting up devices


## 9. Model Fine-Tuning Execution

Trainer displays progress bar (tqdm) per step, logging loss every 20 steps and eval_loss after each epoch.


In [ ]:
train_result = trainer.train()

print("\n=== Training Completed ===")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best eval_loss: {trainer.state.best_metric:.4f}")


[transformers] ***** Running training *****
[transformers]   Num examples = 4,500
[transformers]   Num Epochs = 15
[transformers]   Num update steps per epoch = 141
[transformers]   Instantaneous batch size per device = 4
[transformers]   Total train batch size (w. parallel, distributed & accumulation) = 32
[transformers]   Gradient Accumulation steps = 8
[transformers]   Total optimization steps = 2,115
[transformers]   Number of trainable parameters = 222,903,552


Epoch,Training Loss,Validation Loss
1,7.879369,0.850066
2,6.323200,0.846822
3,5.649554,0.873807
4,4.696143,0.898125
5,4.137735,0.947222


[transformers] 
***** Running Evaluation *****
[transformers]   Num examples = 500
[transformers]   Batch size = 4
[transformers] Saving model checkpoint to /content/t5_paradetox_checkpoints/checkpoint-141
[transformers] Configuration saved in /content/t5_paradetox_checkpoints/checkpoint-141/config.json
[transformers] Configuration saved in /content/t5_paradetox_checkpoints/checkpoint-141/generation_config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Model weights saved in /content/t5_paradetox_checkpoints/checkpoint-141/model.safetensors
[transformers] Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
[transformers] tokenizer config file saved in /content/t5_paradetox_checkpoints/checkpoint-141/tokenizer_config.json
[transformers] 
***** Running Evaluation *****
[transformers]   Num examples = 500
[transformers]   Batch size = 4
[transformers] Saving model checkpoint to /content/t5_paradetox_checkpoints/checkpoint-282
[transformers] Configuration saved in /content/t5_paradetox_checkpoints/checkpoint-282/config.json
[transformers] Configuration saved in /content/t5_paradetox_checkpoints/checkpoint-282/generation_config.json


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Model weights saved in /content/t5_paradetox_checkpoints/checkpoint-564/model.safetensors
[transformers] Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
[transformers] tokenizer config file saved in /content/t5_paradetox_checkpoints/checkpoint-564/tokenizer_config.json
[transformers] 
***** Running Evaluation *****
[transformers]   Num examples = 500
[transformers]   Batch size = 4
[transformers] Saving model checkpoint to /content/t5_paradetox_checkpoints/checkpoint-705
[transformers] Configuration saved in /content/t5_paradetox_checkpoints/checkpoint-705/config.json
[transformers] Configuration saved in /content/t5_paradetox_checkpoints/checkpoint-705/generation_config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Model weights saved in /content/t5_paradetox_checkpoints/checkpoint-705/model.safetensors
[transformers] Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
[transformers] tokenizer config file saved in /content/t5_paradetox_checkpoints/checkpoint-705/tokenizer_config.json
[transformers] 

Training completed. Do not forget to share your model on huggingface.co/models =)


[transformers] Loading best model from /content/t5_paradetox_checkpoints/checkpoint-282 (score: 0.8468219041824341).
[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



=== Training hoan tat ===
Best checkpoint: /content/t5_paradetox_checkpoints/checkpoint-282
Best eval_loss: 0.8468


## 10. Save Best Model to Google Drive

`trainer` automatically loads the best checkpoint into `model` using `load_best_model_at_end=True`.
Saves model weights, config, and tokenizer for independent loading in downstream RL stages.


In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Successfully saved full model and tokenizer to: {OUTPUT_DIR}")
print("Directory contents:")
for f in os.listdir(OUTPUT_DIR):
    path = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  {f}  ({size_mb:.2f} MB)")


[transformers] Saving model checkpoint to /content/drive/MyDrive/Data_Khoaluan/T5
[transformers] Configuration saved in /content/drive/MyDrive/Data_Khoaluan/T5/config.json
[transformers] Configuration saved in /content/drive/MyDrive/Data_Khoaluan/T5/generation_config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Model weights saved in /content/drive/MyDrive/Data_Khoaluan/T5/model.safetensors
[transformers] Saving Trainer.data_collator.tokenizer by default as Trainer.processing_class is `None`
[transformers] tokenizer config file saved in /content/drive/MyDrive/Data_Khoaluan/T5/tokenizer_config.json
[transformers] tokenizer config file saved in /content/drive/MyDrive/Data_Khoaluan/T5/tokenizer_config.json


Da luu full model + tokenizer vao: /content/drive/MyDrive/Data_Khoaluan/T5
Noi dung thu muc:
  config.json  (0.00 MB)
  generation_config.json  (0.00 MB)
  model.safetensors  (850.34 MB)
  tokenizer_config.json  (0.00 MB)
  tokenizer.json  (2.31 MB)
  training_args.bin  (0.01 MB)


## 11. Quick Validation After Fine-Tuning

Generate detoxified samples from validation set for qualitative sanity check.


In [ ]:
model.eval()
model.to(device)

def detoxify(text, max_length=256):
    input_text = TASK_PREFIX + text
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=max_length).to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=max_length, num_beams=1, do_sample=False)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("=== Vi du sinh cau tu tap validation ===\n")
for i in range(5):
    toxic = val_dataset[i]["en_toxic_comment"]
    reference = val_dataset[i]["en_neutral_comment"]
    generated = detoxify(toxic)
    print(f"Toxic     : {toxic}")
    print(f"Reference : {reference}")
    print(f"Generated : {generated}")
    print("-" * 60)


=== Vi du sinh cau tu tap validation ===

Toxic     : no fucking way someone else is going to .
Reference : no way someone else is going to .
Generated : No way someone else is going to.
------------------------------------------------------------
Toxic     : oh fuck off , i 'm outa here .
Reference : i 'm outa here .
Generated : I'm out here.
------------------------------------------------------------
Toxic     : i am sick of hearing about this old fuck who doesn 't know how to groom himself .
Reference : I am sick of hearing about this old man who doesn't know how to take care of himself.
Generated : i am sick of hearing about this old man who doesn 't know how to groom himself .
------------------------------------------------------------
Toxic     : holy fucking jeebus , that 's good !
Reference : That is good
Generated : That's good.
------------------------------------------------------------
Toxic     : sexpot sharon blocked me ! ? ! what the actual fuck ! ! ! we had a bond
Ref